# Silver — Exchange Rates & OMX

Cleans Bronze market data incrementally: reads only new Bronze rows since the last Silver load, drops null rows, derives ISK/EUR cross-rate, renames columns to clean business names, and writes a production-ready table.

**Source:** `bronze.yahoo_finance_raw`  
**Output:** `silver.exchange_rates` (Delta table)  
**Key derivation:** `iskeur_close = iskusd_close / eurusd_close`  

In [ ]:
if spark.catalog.tableExists("silver.exchange_rates"):
    last_date = spark.sql("SELECT MAX(date) FROM silver.exchange_rates").collect()[0][0]
    df_bronze = spark.sql(f"SELECT * FROM bronze.yahoo_finance_raw WHERE date > '{last_date}'")
    print(f"Incremental load — {df_bronze.count()} new Bronze rows after {last_date}")
else:
    df_bronze = spark.read.format("delta").table("bronze.yahoo_finance_raw")
    print(f"Full load — {df_bronze.count()} Bronze rows")

df_bronze.createOrReplaceTempView("bronze_yahoo_finance_incremental")

In [ ]:
df_silver = spark.sql("""
    SELECT
        date,
        close_eurusd_x   AS eurusd_close,
        close_iskusd_x   AS iskusd_close,
        close_omx        AS omx_close,
        high_eurusd_x    AS eurusd_high,
        high_iskusd_x    AS iskusd_high,
        high_omx         AS omx_high,
        low_eurusd_x     AS eurusd_low,
        low_iskusd_x     AS iskusd_low,
        low_omx          AS omx_low,
        open_eurusd_x    AS eurusd_open,
        open_iskusd_x    AS iskusd_open,
        open_omx         AS omx_open,
        volume_eurusd_x  AS eurusd_volume,
        volume_iskusd_x  AS iskusd_volume,
        volume_omx       AS omx_volume,
        ingested_at,
        CASE
            WHEN close_eurusd_x != 0
            THEN ROUND(close_iskusd_x / close_eurusd_x, 6)
            ELSE NULL
        END                  AS iskeur_close,
        CURRENT_TIMESTAMP()  AS processed_at
    FROM bronze_yahoo_finance_incremental
    WHERE close_eurusd_x IS NOT NULL
      AND close_iskusd_x IS NOT NULL
      AND close_eurusd_x != 0
      AND close_iskusd_x != 0
""")

if df_silver.isEmpty():
    raise ValueError("[silver_yahoo_finance] No rows after cleaning - halting.")

df_silver.createOrReplaceTempView("v_fx_silver")

In [ ]:
if not spark.catalog.tableExists("silver.exchange_rates"):
    spark.sql("SELECT * FROM v_fx_silver").write.format("delta").saveAsTable("silver.exchange_rates")
else:
    spark.sql("""
        MERGE INTO silver.exchange_rates AS target
        USING v_fx_silver AS source
        ON target.date = source.date
        WHEN MATCHED THEN UPDATE SET
            target.eurusd_close   = source.eurusd_close,
            target.iskusd_close   = source.iskusd_close,
            target.omx_close      = source.omx_close,
            target.eurusd_high    = source.eurusd_high,
            target.iskusd_high    = source.iskusd_high,
            target.omx_high       = source.omx_high,
            target.eurusd_low     = source.eurusd_low,
            target.iskusd_low     = source.iskusd_low,
            target.omx_low        = source.omx_low,
            target.eurusd_open    = source.eurusd_open,
            target.iskusd_open    = source.iskusd_open,
            target.omx_open       = source.omx_open,
            target.eurusd_volume  = source.eurusd_volume,
            target.iskusd_volume  = source.iskusd_volume,
            target.omx_volume     = source.omx_volume,
            target.ingested_at    = source.ingested_at,
            target.iskeur_close   = source.iskeur_close,
            target.processed_at   = source.processed_at
        WHEN NOT MATCHED THEN INSERT (
            date, eurusd_close, iskusd_close, omx_close,
            eurusd_high, iskusd_high, omx_high,
            eurusd_low, iskusd_low, omx_low,
            eurusd_open, iskusd_open, omx_open,
            eurusd_volume, iskusd_volume, omx_volume,
            ingested_at, iskeur_close, processed_at
        )
        VALUES (
            source.date, source.eurusd_close, source.iskusd_close, source.omx_close,
            source.eurusd_high, source.iskusd_high, source.omx_high,
            source.eurusd_low, source.iskusd_low, source.omx_low,
            source.eurusd_open, source.iskusd_open, source.omx_open,
            source.eurusd_volume, source.iskusd_volume, source.omx_volume,
            source.ingested_at, source.iskeur_close, source.processed_at
        )
    """)

print(f"Saved to silver.exchange_rates - {spark.table('silver.exchange_rates').count()} rows")